In [ ]:
import sys, json, logging, pickle, time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.metrics import (confusion_matrix, roc_auc_score, accuracy_score,
                             precision_score, recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import LeaveOneGroupOut

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors, load_best_combination
from src.feature_extraction import compute_features

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
FIG_DIR = PROJ_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
# Load labelled data
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
best_combo = load_best_combination(DATA_DIR)

# Load features (numeric-only)
X_list = []
for name in best_combo:
    df = pd.read_parquet(DATA_DIR / f"features_{name}.parquet").drop(columns=['organism'], errors='ignore')
    X_list.append(df)
X_full = pd.concat(X_list, axis=1)
train_feature_cols = X_full.columns.tolist()

# Load train/test split
with open(DATA_DIR / 'train_test_indices.json', 'r') as f:
    split = json.load(f)
train_idx, test_idx = split['train_idx'], split['test_idx']

In [ ]:
# Hold‑out metrics
def compute_metrics(y_true, y_pred, y_proba=None):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size==4 else (0,0,0,0)
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Sensitivity': recall_score(y_true, y_pred, pos_label=1),
        'Specificity': tn/(tn+fp) if (tn+fp)>0 else 0,
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_proba) if y_proba is not None else None
    }

metrics_list = []
for s in active_stressors:
    pred_path = MODEL_DIR / f"stressor_{s}_predictions.parquet"
    if pred_path.exists():
        pred = pd.read_parquet(pred_path)
        metrics_list.append({'Stressor': s, **compute_metrics(pred['y_test'], pred['y_pred'], pred.get('y_proba'))})
perf = pd.DataFrame(metrics_list)
print(perf.round(4).to_string(index=False))
perf.to_csv(DATA_DIR / 'final_model_metrics.csv', index=False)

In [ ]:
from sklearn.metrics import precision_recall_curve

print("\n=== Per‑stressor adaptive thresholds ===")
adaptive_metrics = []
for s in active_stressors:
    pred_path = MODEL_DIR / f"stressor_{s}_predictions.parquet"
    if not pred_path.exists():
        continue
    pred = pd.read_parquet(pred_path)
    y_true = pred['y_test'].values
    y_proba = pred['y_proba'].values

    if y_true.sum() == 0:
        continue  # no positives in test set

    prec, rec, thresh = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresh[best_idx]

    y_pred_adapt = (y_proba >= best_thresh).astype(int)
    adaptive_metrics.append({
        'Stressor': s,
        'Threshold': best_thresh,
        'AUC': roc_auc_score(y_true, y_proba),
        'Sensitivity': recall_score(y_true, y_pred_adapt, zero_division=0),
        'Specificity': (y_pred_adapt[y_true==0]==0).mean(),
        'F1': f1_scores[best_idx],
        'MCC': matthews_corrcoef(y_true, y_pred_adapt),
        'Pos rate': y_true.mean()
    })

adapt_df = pd.DataFrame(adaptive_metrics).set_index('Stressor')
print(adapt_df.round(4).to_string())
adapt_df.to_csv(DATA_DIR / 'adaptive_metrics.csv')

In [ ]:
# Taxonomic novelty (requires Spark)
try:
    spark = get_spark_session()
    strain_df = spark.table("enigma_genome_depot_enigma.browser_strain").select("full_name","order","taxon_id").distinct().toPandas()
    strain_df['genus'] = strain_df['full_name'].str.split().str[0]
except Exception:
    strain_df = pd.DataFrame(columns=['full_name','order','taxon_id','genus'])

train_orgs = set(labeled_pd.iloc[train_idx]['organism'])
test_orgs = set(labeled_pd.iloc[test_idx]['organism'])
org_tax = strain_df[strain_df['full_name'].isin(train_orgs|test_orgs)].copy()
org_tax['set'] = org_tax['full_name'].apply(lambda x: 'train' if x in train_orgs else 'test')

for level in ['genus','order']:
    train_taxa = set(org_tax[org_tax['set']=='train'][level].dropna().unique())
    test_taxa = org_tax[org_tax['set']=='test'][level].dropna()
    unseen = test_taxa[~test_taxa.isin(train_taxa)]
    print(f"{level}: {len(unseen)}/{len(test_taxa)} unseen ({len(unseen)/max(1,len(test_taxa)):.1%})")

In [ ]:
# LOGO (Leave-One-Genus-Out) cross-validation
genus_counts = labeled_pd['genus'].value_counts() if 'genus' in labeled_pd.columns else \
    labeled_pd['organism'].str.split().str[0].value_counts()
if 'genus' not in labeled_pd.columns:
    labeled_pd['genus'] = labeled_pd['organism'].str.split().str[0]
groups = labeled_pd['genus']
genus_to_keep = genus_counts[genus_counts>=5].index.tolist()

def run_logo(stressor, X, y_all, groups, checkpoint_dir, iterations=100):
    ckpt = checkpoint_dir / f"logo_{stressor}.parquet"
    if ckpt.exists():
        res = pd.read_parquet(ckpt)
        completed = set(res['genus'])
    else:
        res = pd.DataFrame(columns=['genus','mcc','auc','n_test'])
        completed = set()
    y = y_all[stressor]
    logo = LeaveOneGroupOut()
    for tr, te in logo.split(X, y, groups=groups):
        gen = groups.iloc[te].iloc[0]
        if gen not in genus_to_keep or gen in completed: 
            continue
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        
        # Skip if either test or training set has only one class
        if y_te.nunique()<2 or y_tr.nunique()<2: 
            log.debug(f"{stressor} / {gen}: skipping (single class in {'test' if y_te.nunique()<2 else 'train'} set)")
            continue
        
        m = CatBoostClassifier(iterations=iterations, learning_rate=0.05, depth=4,
                               cat_features=None, eval_metric='F1', 
                               auto_class_weights='Balanced',
                               random_seed=42, verbose=False)
        m.fit(X_tr, y_tr, verbose=False)
        y_pred = m.predict(X_te)
        y_proba = m.predict_proba(X_te)[:,1]
        res = pd.concat([res, pd.DataFrame([{'genus':gen, 'mcc':matthews_corrcoef(y_te,y_pred),
                                             'auc':roc_auc_score(y_te,y_proba), 'n_test':len(y_te)}])],
                        ignore_index=True)
        res.to_parquet(ckpt)
    return res

logo_checkpoint_dir = DATA_DIR / 'logo_checkpoints'
logo_checkpoint_dir.mkdir(exist_ok=True)
logo_results = {}
for s in active_stressors:
    log.info(f"LOGO {s}")
    logo_results[s] = run_logo(s, X_full, labeled_pd, groups, logo_checkpoint_dir, iterations=100)